# HyperCrop — Plant Disease CNN Training (Full Dataset)

Trains MobileNetV2 on the full PlantVillage dataset (38 classes, ~54k images).

**Steps:**
1. `Runtime → Change runtime type → T4 GPU`
2. Fill in your Kaggle credentials in Cell 2
3. `Runtime → Run all`
4. Download `model.zip` at the end → extract into `HyperCorp/backend/model/`

**Expected time:** ~25–35 min on T4 GPU

In [ ]:
# Cell 1 — Check GPU + install extras
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'✓ GPU: {gpus[0].name}')
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
else:
    print('⚠ No GPU — go to Runtime → Change runtime type → T4 GPU')

print(f'TensorFlow: {tf.__version__}')
!pip install tf2onnx onnx onnxruntime kaggle -q
print('✓ Packages ready')

In [ ]:
# Cell 2 — Kaggle credentials
import os, json

KAGGLE_USERNAME = 'YOUR_KAGGLE_USERNAME'  # e.g. ojaswalke356
KAGGLE_KEY      = 'YOUR_KAGGLE_API_KEY'   # KGAT_... token

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('✓ Credentials saved')

In [ ]:
# Cell 3 — Download + extract full PlantVillage (all 38 classes)
import kaggle, zipfile, pathlib, shutil

RAW_DIR     = '/content/raw'
DATASET_DIR = '/content/plantvillage'

os.makedirs(RAW_DIR, exist_ok=True)

print('Downloading PlantVillage (~1.2 GB)...')
kaggle.api.authenticate()
kaggle.api.dataset_download_files('emmarex/plantdisease', path=RAW_DIR, unzip=False)

zip_file = list(pathlib.Path(RAW_DIR).glob('*.zip'))[0]
print(f'Extracting {zip_file.name}...')
with zipfile.ZipFile(zip_file, 'r') as z:
    z.extractall(RAW_DIR)

# ── Find the folder that has the MOST class subdirectories ──
# (fixes the issue where only a subfolder with 15 classes was picked)
best, best_count = None, 0
for p in pathlib.Path(RAW_DIR).rglob('*'):
    if p.is_dir():
        subdirs = [x for x in p.iterdir() if x.is_dir()]
        if len(subdirs) > best_count:
            best, best_count = p, len(subdirs)

print(f'Best folder: {best}  ({best_count} subdirs)')

if os.path.exists(DATASET_DIR):
    shutil.rmtree(DATASET_DIR)
shutil.copytree(str(best), DATASET_DIR)

classes = sorted([p.name for p in pathlib.Path(DATASET_DIR).iterdir() if p.is_dir()])
print(f'\n✓ {len(classes)} classes loaded:')
for c in classes:
    count = len(list((pathlib.Path(DATASET_DIR) / c).glob('*')))
    print(f'  {c}  ({count} images)')

In [ ]:
# Cell 4 — Data pipelines
import numpy as np

IMG_SIZE         = (224, 224)
BATCH_SIZE       = 32
SEED             = 42

full_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    image_size=IMG_SIZE,
    batch_size=None,
    label_mode='categorical',
    shuffle=True,
    seed=SEED,
)

class_names = full_ds.class_names
n_classes   = len(class_names)
total       = sum(1 for _ in full_ds)

n_val   = int(total * 0.15)
n_test  = int(total * 0.10)
n_train = total - n_val - n_test

print(f'Total: {total}  |  Classes: {n_classes}  |  Train/Val/Test: {n_train}/{n_val}/{n_test}')

augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical'),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomBrightness(0.1),
])

AUTOTUNE = tf.data.AUTOTUNE

def prepare(ds, aug=None):
    if aug:
        ds = ds.map(lambda x, y: (aug(x, training=True), y), num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = prepare(full_ds.take(n_train), augment)
val_ds   = prepare(full_ds.skip(n_train).take(n_val))
test_ds  = prepare(full_ds.skip(n_train + n_val))
print('✓ Pipelines ready')

In [ ]:
# Cell 5 — Build MobileNetV2
base = tf.keras.applications.MobileNetV2(
    input_shape=(*IMG_SIZE, 3), include_top=False, weights='imagenet'
)
base.trainable = False

inputs  = tf.keras.Input(shape=(*IMG_SIZE, 3), name='image_input')
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
x = tf.keras.layers.Dense(256, activation='relu')(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(n_classes, activation='softmax', name='predictions')(x)

model = tf.keras.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)
print(f'✓ Model ready — {n_classes} classes, {model.count_params():,} params')

In [ ]:
# Cell 6 — Phase 1: train head (base frozen)
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True, monitor='val_accuracy'),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2, monitor='val_loss', verbose=1),
]

print('Phase 1 — frozen base, 5 epochs max...')
h1 = model.fit(train_ds, validation_data=val_ds, epochs=5, callbacks=callbacks)
print(f'✓ Phase 1 done — best val_acc: {max(h1.history["val_accuracy"]):.4f}')

In [ ]:
# Cell 7 — Phase 2: fine-tune top 30 layers
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

print('Phase 2 — fine-tuning top 30 layers, 10 epochs max...')
h2 = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=callbacks)
print(f'✓ Phase 2 done — best val_acc: {max(h2.history["val_accuracy"]):.4f}')

In [ ]:
# Cell 8 — Evaluate
loss, acc = model.evaluate(test_ds, verbose=1)
print(f'\n✓ Test accuracy: {acc * 100:.2f}%  |  Loss: {loss:.4f}')

In [ ]:
# Cell 9 — Save .h5 + class_indices.json
os.makedirs('/content/model', exist_ok=True)

model.save('/content/model/plant_disease_model.h5')
print('✓ Saved .h5')

with open('/content/model/class_indices.json', 'w') as f:
    json.dump({str(i): name for i, name in enumerate(class_names)}, f, indent=2)
print(f'✓ Saved class_indices.json  ({n_classes} classes)')

In [ ]:
# Cell 10 — Convert to ONNX
import tf2onnx, onnx, onnxruntime as ort

print('Converting to ONNX...')
input_sig = [tf.TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='image_input')]
onnx_model, _ = tf2onnx.convert.from_keras(model, input_signature=input_sig, opset=13)
onnx.save(onnx_model, '/content/model/plant_disease_model.onnx')

# Sanity check
sess = ort.InferenceSession('/content/model/plant_disease_model.onnx')
out  = sess.run(None, {'image_input': np.zeros((1,224,224,3), dtype=np.float32)})
print(f'✓ ONNX ready — output shape: {out[0].shape}')

In [ ]:
# Cell 11 — Download
# After download: extract into HyperCorp/backend/model/
# You need: plant_disease_model.onnx + class_indices.json
import shutil
from google.colab import files

shutil.make_archive('/content/model', 'zip', '/content/model')
files.download('/content/model.zip')
print('✓ Download started')